# 07 — Top Products per Branch

## Purpose
This is the **final step before modeling**. We take the processed branch datasets
from `06_feature_engineering.ipynb` and produce clean, focused files that contain
**only the top 5 products per branch** — the exact products the forecasting models
need to predict.

## Why do we need this step?
The `data/processed/` files contain up to 83 products per branch. The business requirement
is to predict only the **top 5** per branch per day. Training separate models for all 83 products
would be wasteful and potentially noisier.

By isolating the top 5 here, we:
1. Define explicitly which products are in scope for forecasting
2. Produce a clean input file that modeling notebooks can read directly
3. Create a reference table of top products per branch for stakeholder reporting

## Input
7 files in `data/processed/` — produced by `06_feature_engineering.ipynb`

## Output
- `data/processed/top5_per_branch.csv` — one file with all branches, filtered to top 5 products
- `data/processed/top5_summary.csv` — summary table: branch × top 5 products with their volumes

## Run order
Run after `06_feature_engineering.ipynb`.

In [1]:
import os

# ─── PATH CONFIGURATION ───────────────────────────────────────────────
# Option A — After cloning the repo (default, USE_GITHUB = False)
#   git clone https://github.com/DiegoLarrieta/PanemReto
#   cd PanemReto/notebooks
#   jupyter notebook
#   Paths resolve automatically — no changes needed.
#
# Option B — Read directly from GitHub, e.g. Google Colab (USE_GITHUB = True)
#   Works for notebooks 07 (processed branch CSVs are on GitHub).
#   Notebooks 00-06 need the intermediate files which are NOT in the repo
#   (too large). Run 00_data_pipeline.ipynb locally first to generate them.
# ──────────────────────────────────────────────────────────────────────────

USE_GITHUB   = False
GITHUB_BASE  = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — generate locally with 00_data_pipeline.ipynb
else:
    PROJECT_ROOT     = os.path.abspath(os.path.join(os.getcwd(), ".."))
    PROCESSED_DIR    = os.path.join(PROJECT_ROOT, "data", "processed")
    WEATHER_DIR      = os.path.join(PROJECT_ROOT, "data", "weather")
    RAW_DIR          = os.path.join(PROJECT_ROOT, "data", "raw", "Complete Data")
    INTERMEDIATE_DIR = os.path.join(PROJECT_ROOT, "data", "intermediate")

## Step 1 — Imports and paths

In [2]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = PROCESSED_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

## Step 2 — Load all branch files and combine

We load all 7 branch CSVs and concatenate them into one dataframe.
This makes it easy to compute rankings across all branches in one operation.

In [3]:
branch_files = {
    "Panem - Carreta"           : "branch_carreta.csv",
    "Panem - Credi Club"        : "branch_credi_club.csv",
    "Panem - Hospital Zambrano" : "branch_hospital_zambrano.csv",
    "Panem - Hotel Kavia"       : "branch_hotel_kavia.csv",
    "Panem - Plaza Nativa"      : "branch_plaza_nativa.csv",
    "Panem - Plaza QIN"         : "branch_plaza_qin.csv",
    "Panem - Punto Valle"       : "branch_punto_valle.csv",
}

dfs = []
for branch, filename in branch_files.items():
    path = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(path):
        df_branch = pd.read_csv(path, low_memory=False)
        df_branch["operating_date"] = pd.to_datetime(df_branch["operating_date"], errors="coerce")
        df_branch["quantity"] = pd.to_numeric(df_branch["quantity"], errors="coerce").fillna(0)
        dfs.append(df_branch)
        print(f"  Loaded {branch}: {len(df_branch):,} rows, {df_branch['item'].nunique()} products")
    else:
        print(f"  WARNING: {filename} not found. Run 06_feature_engineering.ipynb first.")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_all):,}")
print(f"Total branches: {df_all['sucursal'].nunique()}")
print(f"Total unique products: {df_all['item'].nunique()}")

  Loaded Panem - Carreta: 30,362 rows, 74 products
  Loaded Panem - Credi Club: 5,821 rows, 53 products
  Loaded Panem - Hospital Zambrano: 44,132 rows, 70 products
  Loaded Panem - Hotel Kavia: 41,931 rows, 71 products
  Loaded Panem - Plaza Nativa: 19,325 rows, 76 products
  Loaded Panem - Plaza QIN: 55,336 rows, 77 products
  Loaded Panem - Punto Valle: 63,664 rows, 75 products

Total rows: 260,571
Total branches: 7
Total unique products: 81


## Step 3 — Rank products per branch by cumulative volume

For each branch, we sum the total quantity sold across the entire data range
for each product. Then we rank products from most to least sold.

**Why use total cumulative volume for ranking?**
A product that sells consistently every day ranks higher than one that sold a lot
in a single spike. Cumulative volume over the full period is the most stable indicator
of a product's structural importance to a branch.

In [4]:
# Sum total quantity per (branch, product) over the entire dataset
branch_product_volume = (
    df_all.groupby(["sucursal", "item"], as_index=False)["quantity"]
          .sum()
          .rename(columns={"quantity": "total_qty"})
)

# Rank within each branch (1 = highest volume)
branch_product_volume["rank"] = (
    branch_product_volume.groupby("sucursal")["total_qty"]
                         .rank(method="first", ascending=False)
                         .astype(int)
)

# Show the top 10 per branch as a summary table
branch_product_volume_sorted = branch_product_volume.sort_values(["sucursal", "rank"])

for branch in sorted(branch_product_volume_sorted["sucursal"].unique()):
    print(f"\n{'='*55}\n  {branch}\n{'='*55}")
    t = branch_product_volume_sorted[
        (branch_product_volume_sorted["sucursal"] == branch) &
        (branch_product_volume_sorted["rank"] <= 10)
    ][["rank", "item", "total_qty"]].reset_index(drop=True)
    display(t)


  Panem - Carreta


,rank,item,total_qty
0,1,CONCHA VAINILLA,23414.0
1,2,CONCHA CHOCOLATE,12153.0
2,3,GALLETA DE AVENA,6545.0
3,4,MUFFIN PLATANO VEGANO,5756.0
4,5,MUFFIN BLUEBERRY,5550.0
5,6,PAIN AU CHOCOLAT,5340.0
6,7,ENSALADA CESAR,4782.0
7,8,CROISSANT ALMENDRAS,4686.0
8,9,BAGUETTE CAPRESE,4683.0
9,10,SANDWICH ENSALADA POLLO,4622.0



  Panem - Credi Club


,rank,item,total_qty
0,1,CONCHA VAINILLA,2680.0
1,2,CONCHA CHOCOLATE,1455.0
2,3,CHILAQUILES PANEM,1340.0
3,4,CONCHA UBER,679.0
4,5,ENCHILADAS SUIZAS,673.0
5,6,PAIN AU CHOCOLAT,466.0
6,7,MUFFIN PLATANO VEGANO,419.0
7,8,CROISSANT DE JAMÓN Y QUESO,418.0
8,9,ROL DE CANELA,394.0
9,10,MUFFIN BLUEBERRY,393.0



  Panem - Hospital Zambrano


...

,rank,item,total_qty
0,1,CONCHA VAINILLA,25498.0
1,2,CONCHA CHOCOLATE,11423.0
2,3,CHILAQUILES PANEM,4633.0
3,4,PAIN AU CHOCOLAT,3318.0
4,5,OREJA NATURAL,3215.0
5,6,CROISSANT ALMENDRAS,3189.0
6,7,MUFFIN BLUEBERRY,3176.0
7,8,GALLETA DE AVENA,2994.0
8,9,VASO BIO,2846.0
9,10,MUFFIN PLATANO VEGANO,2796.0



  Panem - Plaza QIN


,rank,item,total_qty
0,1,CONCHA VAINILLA,61794.0
1,2,CONCHA CHOCOLATE,22826.0
2,3,BAGUETTE PETITE MAYOREO 10,20326.0
3,4,CHILAQUILES PANEM,14705.0
4,5,CROISSANT ALMENDRAS,11159.0
5,6,GALLETA DE AVENA,9248.0
6,7,PAIN AU CHOCOLAT,9088.0
7,8,OREJA NATURAL,8911.0
8,9,MUFFIN BLUEBERRY,7861.0
9,10,CROISSANT DE JAMÓN Y QUESO,7006.0



  Panem - Punto Valle


,rank,item,total_qty
0,1,CONCHA VAINILLA,55903.0
1,2,CONCHA CHOCOLATE,22955.0
2,3,CHILAQUILES PANEM,21201.0
3,4,CROISSANT ALMENDRAS,16965.0
4,5,PAIN AU CHOCOLAT,13382.0
5,6,ROL DE CANELA,11900.0
6,7,MUFFIN BLUEBERRY,11778.0
7,8,ENCHILADAS SUIZAS,10698.0
8,9,MUFFIN PLATANO VEGANO,9491.0
9,10,GALLETA DE AVENA,9317.0


## Step 4 — Identify the top 5 products per branch

We keep only the products with rank 1–5 per branch.
This is the official top-5 list that the forecasting models will be trained on.

In [5]:
# Get the top 5 product names per branch
top5_per_branch = branch_product_volume[branch_product_volume["rank"] <= 5].copy()

print("TOP 5 PRODUCTS PER BRANCH")
print("=" * 50)
for branch in sorted(top5_per_branch["sucursal"].unique()):
    products = (
        top5_per_branch[top5_per_branch["sucursal"] == branch]
        .sort_values("rank")[["rank", "item", "total_qty"]]
    )
    print(f"\n{branch}:")
    for _, row in products.iterrows():
        print(f"  #{int(row['rank'])}  {row['item']}  ({row['total_qty']:,.0f} units)")

TOP 5 PRODUCTS PER BRANCH

Panem - Carreta:
  #1  CONCHA VAINILLA  (23,414 units)
  #2  CONCHA CHOCOLATE  (12,153 units)
  #3  GALLETA DE AVENA  (6,545 units)
  #4  MUFFIN PLATANO VEGANO  (5,756 units)
  #5  MUFFIN BLUEBERRY  (5,550 units)

Panem - Credi Club:
  #1  CONCHA VAINILLA  (2,680 units)
  #2  CONCHA CHOCOLATE  (1,455 units)
  #3  CHILAQUILES PANEM  (1,340 units)
  #4  CONCHA UBER  (679 units)
  #5  ENCHILADAS SUIZAS  (673 units)

Panem - Hospital Zambrano:
  #1  CONCHA VAINILLA  (49,745 units)
  #2  CONCHA CHOCOLATE  (20,844 units)
  #3  CROISSANT DE JAMÓN Y QUESO  (11,529 units)
  #4  OREJA NATURAL  (10,140 units)
  #5  CROISSANT ALMENDRAS  (10,089 units)

Panem - Hotel Kavia:
  #1  CHILAQUILES PANEM  (38,243 units)
  #2  ENCHILADAS SUIZAS  (18,897 units)
  #3  CONCHA VAINILLA  (17,759 units)
  #4  REBANADA CHEESECAKE TORTUGA  (13,618 units)
  #5  CONCHA CHOCOLATE  (11,454 units)

Panem - Plaza Nativa:
  #1  CONCHA VAINILLA  (25,498 units)
  #2  CONCHA CHOCOLATE  (11,423 uni

## Step 5 — Filter the full dataset to top 5 products only

We create a new dataframe that contains ONLY the rows for the top 5 products at each branch.
This is what the model will train on.

In [6]:
# Create a set of (branch, item) pairs that are in the top 5
top5_pairs = set(
    zip(top5_per_branch["sucursal"], top5_per_branch["item"])
)

# Filter the main dataframe to those pairs only
df_top5 = df_all[
    df_all.apply(lambda row: (row["sucursal"], row["item"]) in top5_pairs, axis=1)
].copy()

print(f"Rows with all products:      {len(df_all):,}")
print(f"Rows with only top 5:        {len(df_top5):,}")
print(f"Branches:                    {df_top5['sucursal'].nunique()}")
print(f"Unique products in dataset:  {df_top5['item'].nunique()}")

Rows with all products:      260,571
Rows with only top 5:        33,118
Branches:                    7
Unique products in dataset:  14


## Step 6 — Save the outputs

We save two files:
1. **`top5_per_branch.csv`** — the full filtered dataset (all rows, only top-5 products)
   This is the direct input for the modeling notebooks in `models/`.
2. **`top5_summary.csv`** — a compact reference table showing the top-5 ranking per branch
   Useful for business reporting and for understanding model scope.

In [10]:
# Save the full filtered dataset
output_path = os.path.join(PROCESSED_DIR, "top5_per_branch.csv")
df_top5.to_csv(output_path, index=False)
print(f"Saved top5_per_branch.csv  ({len(df_top5):,} rows)")

# Save the summary table
summary_path = os.path.join(PROCESSED_DIR, "top5_summary.csv")
top5_per_branch.sort_values(["sucursal", "rank"]).to_csv(summary_path, index=False)
print(f"Saved top5_summary.csv     ({len(top5_per_branch)} rows — 7 branches × 5 products)")

Saved top5_per_branch.csv  (33,118 rows)
Saved top5_summary.csv     (35 rows — 7 branches × 5 products)


## Step 7 — Sanity check on date coverage per product

Before passing data to any model, we check: how many days of history does each
top-5 product have per branch? Too few days = the model may overfit or produce
unreliable predictions.

In [11]:
date_coverage = (
    df_top5.groupby(["sucursal", "item"])
           .agg(
               first_sale=("operating_date", "min"),
               last_sale=("operating_date", "max"),
               days_with_sales=("operating_date", "count")
           )
           .reset_index()
)
date_coverage["date_range_days"] = (date_coverage["last_sale"] - date_coverage["first_sale"]).dt.days

for branch in sorted(date_coverage["sucursal"].unique()):
    print(f"\n{branch}:")
    t = date_coverage[date_coverage["sucursal"] == branch][
        ["item", "first_sale", "last_sale", "days_with_sales", "date_range_days"]
    ].reset_index(drop=True)
    display(t)


Panem - Carreta:


,item,first_sale,last_sale,days_with_sales,date_range_days
0,CONCHA CHOCOLATE,2022-02-18,2026-02-12,942,1455
1,CONCHA VAINILLA,2022-02-18,2026-02-12,944,1455
2,GALLETA DE AVENA,2022-02-18,2026-02-12,948,1455
3,MUFFIN BLUEBERRY,2022-02-21,2026-02-12,910,1452
4,MUFFIN PLATANO VEGANO,2022-02-21,2026-02-12,884,1452



Panem - Credi Club:


,item,first_sale,last_sale,days_with_sales,date_range_days
0,CHILAQUILES PANEM,2024-11-21,2026-02-11,338,447
1,CONCHA CHOCOLATE,2024-11-22,2026-02-12,301,447
2,CONCHA UBER,2024-11-29,2026-01-26,159,423
3,CONCHA VAINILLA,2024-11-21,2026-02-12,370,448
4,ENCHILADAS SUIZAS,2024-11-22,2026-02-12,244,447



Panem - Hospital Zambrano:


...

,item,first_sale,last_sale,days_with_sales,date_range_days
0,CHILAQUILES PANEM,2024-02-21,2026-02-12,639,722
1,CONCHA CHOCOLATE,2024-02-21,2026-02-12,652,722
2,CONCHA VAINILLA,2024-02-21,2026-02-12,658,722
3,OREJA NATURAL,2024-02-22,2026-02-12,559,721
4,PAIN AU CHOCOLAT,2024-02-21,2026-02-12,644,722



Panem - Plaza QIN:


,item,first_sale,last_sale,days_with_sales,date_range_days
0,BAGUETTE PETITE MAYOREO 10,2022-09-14,2023-07-11,71,300
1,CHILAQUILES PANEM,2022-04-28,2026-02-12,1352,1386
2,CONCHA CHOCOLATE,2022-04-28,2026-02-12,1320,1386
3,CONCHA VAINILLA,2022-04-28,2026-02-12,1349,1386
4,CROISSANT ALMENDRAS,2022-04-28,2026-02-12,1337,1386



Panem - Punto Valle:


,item,first_sale,last_sale,days_with_sales,date_range_days
0,CHILAQUILES PANEM,2022-02-01,2026-02-12,1467,1472
1,CONCHA CHOCOLATE,2022-02-01,2026-02-12,1461,1472
2,CONCHA VAINILLA,2022-02-01,2026-02-12,1469,1472
3,CROISSANT ALMENDRAS,2022-02-01,2026-02-12,1461,1472
4,PAIN AU CHOCOLAT,2022-02-01,2026-02-12,1462,1472


## Summary

| Output file | Location | Contents |
|---|---|---|
| `top5_per_branch.csv` | `data/processed/` | Full dataset filtered to top 5 products per branch — input for models |
| `top5_summary.csv` | `data/processed/` | Reference table: 7 branches × 5 products with total volumes |

**You are now ready to train models.**

The modeling notebooks live in `models/arima/`, `models/prophet/`, and `models/xgboost/`.
Each should read `data/processed/top5_per_branch.csv` (or the individual branch files)
and train a separate model per (branch, product) combination.